# Bengaluru Climate Risk Scanner
## Notebook 03: Extreme-heat model (Phase 3)

This notebook maps where Bengaluru gets hottest in summer and turns that into **Low / Medium / High heat zones**, the second hazard in the property risk score.

**Method**
1. Collect every clear **Landsat 8 and 9** image of Bengaluru from the hot season (March to May, 2022 to 2024) in Google Earth Engine.
2. Convert the thermal band to **land surface temperature (LST)** in °C, and the red and near-infrared bands to **NDVI** (how green the surface is).
3. Take the **median** of all clear images, so one unusual day does not dominate.
4. Combine: **heat index = 0.7 × normalised LST + 0.3 × (1 − normalised NDVI)**. Hot, bare, concrete areas score highest.
5. Split city land into classes (the hottest 15% is High), make polygons, and summarise by ward.

**Outputs** (in `MyDrive/bengaluru-climate-risk/data/processed/`)

| File | What it holds | Used by |
|---|---|---|
| `heat_risk.tif` | Heat index, 0 to 1 | GeoServer WMS (colour ramp) |
| `heat_class.tif` | 1 Low, 2 Medium, 3 High, 0 permanent water | Property scoring checks |
| `heat_values.tif` | Summer LST (°C), NDVI, number of clear images | Popups ("38.4 °C typical summer surface temperature") |
| `bengaluru.gpkg` → `heat_zones` | Low/Medium/High polygons | GeoServer WFS (property scoring) |
| `bengaluru.gpkg` → `ward_heat_summary` | Mean LST and share in High heat, per ward | Ward view |

All outputs sit on exactly the same 30 m grid as the flood model, so the two can be compared cell by cell.

**Before you start:** run Notebooks 01 and 02. This notebook needs your Earth Engine project (same login as Notebook 01).

## 1. Install libraries

In [ ]:
!pip install -q geemap rasterio

## 2. Connect Google Drive and load the project configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil

PROJECT_DIR = '/content/drive/MyDrive/bengaluru-climate-risk'
with open(f'{PROJECT_DIR}/config.json') as f:
    CONFIG = json.load(f)

RAW = CONFIG['paths']['raw']
OUT = CONFIG['paths']['processed']
GPKG = CONFIG['paths']['gpkg']
WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)

print('City:', CONFIG['city'])
print('Hot season:', CONFIG['heat_season'])

## 3. Connect to Google Earth Engine
Uses the project ID saved in `config.json`. If the sign-in pop-up fails in Colab ("credential propagation was unsuccessful"), set `AUTH_MODE = 'notebook'`: it gives you a link and a code to paste back.

In [ ]:
# [EE]
import ee, geemap

EE_PROJECT = CONFIG['ee_project']
AUTH_MODE = None          # None = Colab pop-up; 'notebook' = link + code

if EE_PROJECT == 'YOUR-EE-PROJECT-ID':
    raise ValueError('Set your Earth Engine project ID in config.json (Notebook 01, section 3).')

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode=AUTH_MODE) if AUTH_MODE else ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine ready:', ee.String('OK').getInfo(), '| project', EE_PROJECT)

## 4. Heat model settings
- `max_cloud_pct`: skip Landsat scenes with more than this share of cloud. Remaining clouds are masked pixel by pixel.
- `weights`: how much surface temperature and lack of vegetation count in the heat index.
- `normalise_percentiles`: LST and NDVI are rescaled to 0–1 using the 2nd and 98th percentile of city land, so a few extreme pixels (a factory roof, a lake edge) don't stretch the scale.
- `class_quantiles`: same as the flood model: the hottest 15% of city land is High, the next 30% Medium.

In [ ]:
HEAT_MODEL = {
    "max_cloud_pct": 20,
    "weights": {"lst": 0.7, "low_ndvi": 0.3},
    "normalise_percentiles": [2, 98],
    "class_quantiles": [0.55, 0.85],
    "min_zone_cells": 10,
}
assert abs(sum(HEAT_MODEL['weights'].values()) - 1) < 1e-9
season = CONFIG['heat_season']
print('Years:', season['years'], '| months', season['start_month'], 'to', season['end_month'])

## 5. Find the summer Landsat images
Landsat 8 (since 2013) and Landsat 9 (since 2021) pass over every 16 days each, so together about every 8 days. The filters keep scenes over Bengaluru, in the chosen years and months, with little cloud.

The printout lists each scene's date and satellite. Expect roughly 10 to 30 scenes; fewer than 5 makes the composite noisy (then raise `max_cloud_pct` or add a year in `config.json`).

In [ ]:
# [EE]
import pandas as pd

import geopandas as gpd
aoi_gdf = gpd.read_file(GPKG, layer='aoi')
w, s, e, n = aoi_gdf.total_bounds
m = CONFIG['buffer_deg']
export_region = ee.Geometry.Rectangle([w - m, s - m, e + m, n + m])

landsat = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
           .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
           .filterBounds(export_region)
           .filter(ee.Filter.calendarRange(min(season['years']), max(season['years']), 'year'))
           .filter(ee.Filter.calendarRange(season['start_month'], season['end_month'], 'month'))
           .filter(ee.Filter.lt('CLOUD_COVER', HEAT_MODEL['max_cloud_pct'])))

props = ['system:time_start', 'SPACECRAFT_ID', 'CLOUD_COVER', 'WRS_PATH', 'WRS_ROW']
rows = landsat.reduceColumns(ee.Reducer.toList(len(props)), props).get('list').getInfo()
scenes = pd.DataFrame(rows, columns=['time', 'satellite', 'cloud_pct', 'path', 'row'])
scenes['date'] = pd.to_datetime(scenes['time'], unit='ms').dt.date
scenes = scenes.drop(columns='time').sort_values('date').reset_index(drop=True)
scenes['cloud_pct'] = scenes['cloud_pct'].round(1)

print(f'{len(scenes)} Landsat scenes found')
print(scenes[['date', 'satellite', 'cloud_pct', 'path', 'row']].to_string(index=False))
scenes.to_csv(f'{RAW}/landsat_scenes_heat.csv', index=False)
if len(scenes) < 5:
    print('\nWARNING: very few scenes. Raise max_cloud_pct or add years to heat_season in config.json.')

## 6. Land surface temperature and NDVI for each image, then the summer median
For every scene:
- **Cloud mask**: the `QA_PIXEL` band flags fill, dilated cloud, cirrus, cloud and cloud shadow (bits 0 to 4). Any flagged pixel is removed.
- **LST**: band `ST_B10` × 0.00341802 + 149.0 gives Kelvin (the official Collection 2 scale factors); − 273.15 gives °C.
- **NDVI** = (NIR − Red) / (NIR + Red), from bands `SR_B5` and `SR_B4` after applying their scale factors (× 0.0000275 − 0.2). It runs from about −1 (water) through 0 (bare, concrete) to +0.8 (dense trees).

Then the **median** of each pixel across all clear scenes gives a typical hot-season value, plus a count of clear observations per pixel.

In [ ]:
# [EE]
def prep(img):
    qa = img.select('QA_PIXEL')
    clear = qa.bitwiseAnd(0b11111).eq(0)                     # bits 0-4: fill, dilated cloud, cirrus, cloud, shadow
    sr = img.select(['SR_B4', 'SR_B5']).multiply(0.0000275).add(-0.2)
    ndvi = sr.normalizedDifference(['SR_B5', 'SR_B4']).rename('ndvi')
    lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('lst_c')
    return lst.addBands(ndvi).updateMask(clear).set('system:time_start', img.get('system:time_start'))

prepared = landsat.map(prep)
lst_med  = prepared.select('lst_c').median().rename('lst_c')
ndvi_med = prepared.select('ndvi').median().rename('ndvi')
n_obs    = prepared.select('lst_c').count().rename('n_obs')

# Quick check over the AOI before downloading
stats = lst_med.reduceRegion(ee.Reducer.percentile([5, 50, 95]), export_region, scale=90, maxPixels=1e9).getInfo()
print('Summer LST over the area (°C): 5th / median / 95th percentile =',
      [round(stats[k], 1) for k in ('lst_c_p5', 'lst_c_p50', 'lst_c_p95')])

## 7. Download the three layers
Each layer is downloaded on the project's 30 m UTM grid and saved to `data/raw`. Earth Engine limits one download to about 48 MB, so the bands are downloaded one at a time.

Note: Landsat's thermal sensor measures at 100 m; USGS delivers it resampled to 30 m. So temperature detail is coarser than the 30 m grid suggests.

In [ ]:
# [EE]
def export_tif(image, name):
    local = f'{WORK}/{name}.tif'
    geemap.ee_export_image(image.toFloat(), filename=local, scale=CONFIG['resolution_m'],
                           crs=CONFIG['analysis_crs'], region=export_region, file_per_band=False)
    shutil.copy(local, f'{RAW}/{name}.tif')
    print('Saved', f'{RAW}/{name}.tif')

export_tif(lst_med, 'heat_lst_raw')
export_tif(ndvi_med, 'heat_ndvi_raw')
export_tif(n_obs, 'heat_nobs_raw')

## 8. Put everything on the flood model's grid
The downloads are warped onto exactly the same grid as `dem_30m.tif` (same rows, columns and position). Then every heat cell matches a flood cell, which the web app and the validation rely on. If the grids already match, nothing changes.

The city, lake and built-up layers from Notebook 01 are loaded on the same grid.

In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.warp import reproject, Resampling

with rasterio.open(f'{RAW}/dem_30m.tif') as src:
    profile, transform, crs = src.profile, src.transform, src.crs
    shape, res = (src.height, src.width), src.res[0]

def on_grid(path):
    """Read a single-band raster and warp it onto the DEM grid (NaN where empty)."""
    with rasterio.open(path) as src:
        out = np.full(shape, np.nan, dtype='float32')
        reproject(source=rasterio.band(src, 1), destination=out,
                  src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                  dst_transform=transform, dst_crs=crs, dst_nodata=np.nan,
                  resampling=Resampling.bilinear)
    return out

lst  = on_grid(f'{RAW}/heat_lst_raw.tif')
ndvi = on_grid(f'{RAW}/heat_ndvi_raw.tif')
nobs = on_grid(f'{RAW}/heat_nobs_raw.tif')
with rasterio.open(f'{RAW}/builtup_30m.tif') as src:
    builtup = src.read(1).astype('float32')

boundary = gpd.read_file(GPKG, layer='bbmp_boundary').to_crs(crs)
lakes    = gpd.read_file(GPKG, layer='lakes').to_crs(crs)

def burn(gdf):
    if len(gdf) == 0:
        return np.zeros(shape, dtype=bool)
    return features.rasterize(((g, 1) for g in gdf.geometry if g is not None and not g.is_empty),
                              out_shape=shape, transform=transform, fill=0, dtype='uint8').astype(bool)

city_mask = burn(boundary)
lake_mask = burn(lakes)
land = city_mask & ~lake_mask & np.isfinite(lst) & np.isfinite(ndvi)
cell_km2 = res * res / 1e6

print(f'Grid {shape[1]} x {shape[0]} at {res:.0f} m')
print(f'City land with heat data: {land.sum() * cell_km2:,.0f} km² '
      f'({100 * land.sum() / (city_mask & ~lake_mask).sum():.1f}% of city land)')
print(f'Clear images per pixel in the city: min {np.nanmin(nobs[land]):.0f}, median {np.nanmedian(nobs[land]):.0f}')
print(f'Summer LST in the city: {np.nanpercentile(lst[land], 5):.1f} to {np.nanpercentile(lst[land], 95):.1f} °C '
      f'(5th to 95th percentile), median {np.nanmedian(lst[land]):.1f} °C')
print(f'NDVI in the city: median {np.nanmedian(ndvi[land]):.2f}')

## 9. Heat index and classes
1. **Normalise** LST and NDVI to 0–1 between their 2nd and 98th percentiles in the city (values beyond are clipped to 0 or 1).
2. **Heat index** = 0.7 × normalised LST + 0.3 × (1 − normalised NDVI). Low NDVI (little vegetation) raises heat risk, because green areas cool through shade and evaporation.
3. **Classes** from the 55th and 85th percentiles of the index: Low, Medium, High. Lakes are class 0 (water).

In [ ]:
import pandas as pd

p_lo, p_hi = HEAT_MODEL['normalise_percentiles']

def normalise(a):
    lo, hi = np.nanpercentile(a[land], [p_lo, p_hi])
    return np.clip((a - lo) / (hi - lo), 0, 1), round(float(lo), 3), round(float(hi), 3)

lst_n, lst_lo, lst_hi = normalise(lst)
ndvi_n, ndvi_lo, ndvi_hi = normalise(ndvi)

wts = HEAT_MODEL['weights']
heat = (wts['lst'] * lst_n + wts['low_ndvi'] * (1 - ndvi_n)).astype('float32')
heat = np.where(land, heat, np.nan).astype('float32')

class_breaks = np.nanquantile(heat, HEAT_MODEL['class_quantiles']).round(3).tolist()
NODATA_CLASS = 255
heat_class = np.full(shape, NODATA_CLASS, dtype='uint8')
heat_class[city_mask & lake_mask] = 0
heat_class[land] = (1 + np.digitize(heat[land], class_breaks)).astype('uint8')

LEVELS = {1: 'Low', 2: 'Medium', 3: 'High'}
summary = pd.DataFrame({
    'area_km2':    [round((heat_class == c).sum() * cell_km2, 1) for c in (1, 2, 3)],
    'mean_lst_c':  [round(float(np.nanmean(lst[heat_class == c])), 1) for c in (1, 2, 3)],
    'mean_ndvi':   [round(float(np.nanmean(ndvi[heat_class == c])), 2) for c in (1, 2, 3)],
}, index=[LEVELS[c] for c in (1, 2, 3)])
summary['share_%'] = (100 * summary['area_km2'] / summary['area_km2'].sum()).round(1)

print(f'LST scaled between {lst_lo} and {lst_hi} °C; NDVI between {ndvi_lo} and {ndvi_hi}')
print(f'Class breaks (heat index): Low < {class_breaks[0]} <= Medium < {class_breaks[1]} <= High')
print(summary.to_string())

## 10. Urban heat island: how much hotter is concrete than green?
A headline number for the report and interviews: the difference in typical summer surface temperature between densely built-up cells (80% or more built-up) and green, open cells (under 20% built-up, NDVI above 0.4).

In [ ]:
dense = land & (builtup >= 0.8)
green = land & (builtup < 0.2) & (ndvi > 0.4)
if dense.sum() and green.sum():
    uhi = float(np.nanmean(lst[dense]) - np.nanmean(lst[green]))
    print(f'Dense built-up cells: {dense.sum() * cell_km2:,.1f} km², mean LST {np.nanmean(lst[dense]):.1f} °C')
    print(f'Green open cells:     {green.sum() * cell_km2:,.1f} km², mean LST {np.nanmean(lst[green]):.1f} °C')
    print(f'Surface urban heat island: {uhi:+.1f} °C')
else:
    uhi = None
    print('Not enough dense or green cells to compare.')

corr = np.corrcoef(lst[land], ndvi[land])[0, 1]
print(f'Correlation between LST and NDVI in the city: {corr:.2f} (negative = greener is cooler)')

## 11. Maps
Check the pattern: dense older areas and industrial zones (for example Peenya in the north-west) should be hot; parks such as Cubbon Park and Lalbagh, and lakes, should be cool.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

extent = [transform.c, transform.c + shape[1] * res, transform.f - shape[0] * res, transform.f]
show = lambda a: np.where(city_mask, a, np.nan)

fig, axes = plt.subplots(2, 2, figsize=(14, 13))
panels = [(show(lst), 'inferno', 'Summer land surface temperature (°C)', np.nanpercentile(lst[land], [2, 98])),
          (show(ndvi), 'YlGn', 'NDVI (greenness)', [0, 0.6]),
          (heat, 'YlOrRd', 'Heat index (0–1)', [0, 1])]
for ax, (arr, cmap, title, (vmin, vmax)) in zip(axes.flat, panels):
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, extent=extent)
    fig.colorbar(im, ax=ax, shrink=0.75); ax.set_title(title)

class_cmap = ListedColormap(['#4a90d9', '#fff5b1', '#fd8d3c', '#bd0026'])
ax = axes.flat[3]
ax.imshow(np.where(heat_class == NODATA_CLASS, np.nan, heat_class), cmap=class_cmap,
          norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4), extent=extent, interpolation='nearest')
ax.set_title('Heat classes')
ax.legend(handles=[Patch(color=c, label=l) for c, l in
                   zip(class_cmap.colors, ['Permanent water', 'Low', 'Medium', 'High'])], loc='lower left')
for ax in axes.flat:
    boundary.boundary.plot(ax=ax, color='black', linewidth=0.7); ax.set_axis_off()
plt.tight_layout(); plt.show()

## 12. Save the rasters

In [ ]:
def save_tif(name, arrays, dtype, nodata, band_names=None):
    arrays = arrays if isinstance(arrays, list) else [arrays]
    prof = profile.copy()
    prof.update(driver='GTiff', dtype=dtype, count=len(arrays), nodata=nodata,
                compress='deflate', tiled=True, blockxsize=256, blockysize=256)
    local = f'{WORK}/{name}'
    with rasterio.open(local, 'w', **prof) as dst:
        for i, a in enumerate(arrays, start=1):
            dst.write(a.astype(dtype), i)
            if band_names:
                dst.set_band_description(i, band_names[i - 1])
    shutil.copy(local, f'{OUT}/{name}')
    print(f'Saved {OUT}/{name}  ({os.path.getsize(local) / 1e6:.1f} MB)')

fill = lambda a: np.where(city_mask & np.isfinite(a), a, -9999)
save_tif('heat_risk.tif', np.where(np.isnan(heat), -9999, heat), 'float32', -9999)
save_tif('heat_class.tif', heat_class, 'uint8', NODATA_CLASS)
save_tif('heat_values.tif', [fill(lst), fill(ndvi), fill(nobs)], 'float32', -9999,
         band_names=['lst_c', 'ndvi', 'n_clear_images'])

## 13. Heat zones as polygons (for WFS)
Same steps as the flood zones: sieve away patches under 0.9 ha, polygonise, add the class, area, mean LST and mean heat index, and convert to latitude/longitude.

Polygon areas per class can differ a little from the raster table in section 9, because sieving merges small patches into their neighbours.

In [ ]:
from shapely.geometry import shape as to_shape

zones_arr = np.where(land, heat_class, 0).astype('uint8')
sieved = features.sieve(zones_arr, size=HEAT_MODEL['min_zone_cells'], connectivity=8)
sieved = np.where(land, sieved, 0).astype('uint8')
sieved[land & (sieved == 0)] = heat_class[land & (sieved == 0)]

polys, classes = [], []
for geom, val in features.shapes(sieved, mask=sieved > 0, transform=transform, connectivity=8):
    polys.append(to_shape(geom)); classes.append(int(val))

heat_zones = gpd.GeoDataFrame({'risk_class': classes}, geometry=polys, crs=crs)
heat_zones['risk_level'] = heat_zones['risk_class'].map(LEVELS)
heat_zones['area_m2'] = heat_zones.area.round(0)

zone_id = features.rasterize(((g, i + 1) for i, g in enumerate(heat_zones.geometry)),
                             out_shape=shape, transform=transform, fill=0, dtype='int32')
inside = (zone_id > 0) & land
counts = np.maximum(np.bincount(zone_id[inside], minlength=len(heat_zones) + 1)[1:], 1)
for col, arr, nd in (('mean_lst_c', lst, 1), ('mean_heat_index', heat, 3)):
    sums = np.bincount(zone_id[inside], weights=arr[inside], minlength=len(heat_zones) + 1)[1:]
    heat_zones[col] = np.round(sums / counts, nd)

heat_zones = heat_zones.to_crs(CONFIG['web_crs'])
print(f'{len(heat_zones):,} polygons')
print(heat_zones.groupby('risk_level').agg(polygons=('area_m2', 'size'), km2=('area_m2', lambda a: round(a.sum() / 1e6, 1)),
                                            mean_lst_c=('mean_lst_c', 'mean')).round(1).to_string())

## 14. Ward summary (optional)

In [ ]:
ward_summary = None
layers = gpd.list_layers(GPKG)['name'].tolist()
if 'wards' in layers:
    wards = gpd.read_file(GPKG, layer='wards').to_crs(crs).reset_index(drop=True)
    ward_id = features.rasterize(((g, i + 1) for i, g in enumerate(wards.geometry)),
                                 out_shape=shape, transform=transform, fill=0, dtype='int32')
    sel = (ward_id > 0) & land
    nw = len(wards) + 1
    total = np.maximum(np.bincount(ward_id[sel], minlength=nw)[1:], 1)
    wards['mean_lst_c'] = np.round(np.bincount(ward_id[sel], weights=lst[sel], minlength=nw)[1:] / total, 1)
    wards['mean_ndvi'] = np.round(np.bincount(ward_id[sel], weights=ndvi[sel], minlength=nw)[1:] / total, 2)
    wards['high_heat_share_pct'] = np.round(
        100 * np.bincount(ward_id[sel], weights=(heat_class[sel] == 3), minlength=nw)[1:] / total, 1)
    ward_summary = wards.to_crs(CONFIG['web_crs'])
    name_col = next((c for c in wards.columns if 'name' in c.lower()), None)
    cols = ([name_col] if name_col else []) + ['mean_lst_c', 'mean_ndvi', 'high_heat_share_pct']
    print('Hottest wards (by mean summer LST):')
    print(ward_summary.sort_values('mean_lst_c', ascending=False)[cols].head(10).to_string(index=False))
else:
    print('No ward layer in the GeoPackage: skipped.')

## 15. Save the polygons to the GeoPackage

In [ ]:
LOCAL_GPKG = f'{WORK}/bengaluru.gpkg'
shutil.copy(GPKG, LOCAL_GPKG)
heat_zones.to_file(LOCAL_GPKG, layer='heat_zones', driver='GPKG')
if ward_summary is not None:
    ward_summary.to_file(LOCAL_GPKG, layer='ward_heat_summary', driver='GPKG')
shutil.copy(LOCAL_GPKG, GPKG)
print('Layers now in the GeoPackage:', gpd.list_layers(GPKG)['name'].tolist())

## 16. Record the model in `config.json`

In [ ]:
from datetime import datetime, timezone

CONFIG['heat_model'] = {
    **HEAT_MODEL,
    'scenes_used': int(len(scenes)) if 'scenes' in globals() else None,
    'lst_scale_c': [lst_lo, lst_hi],
    'ndvi_scale': [ndvi_lo, ndvi_hi],
    'class_breaks_index': class_breaks,
    'class_summary': summary.to_dict(orient='index'),
    'surface_uhi_c': round(uhi, 2) if uhi is not None else None,
    'run_at': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC'),
}
with open(f'{PROJECT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(json.dumps(CONFIG['heat_model'], indent=2))

## 17. Summary

In [ ]:
print('Outputs in', OUT)
for f in sorted(os.listdir(OUT)):
    print(f'  {f:<28} {os.path.getsize(f"{OUT}/{f}") / 1e6:6.1f} MB')
print(f'\nHigh heat: {summary.loc["High", "area_km2"]} km² ({summary.loc["High", "share_%"]}% of city land), '
      f'mean summer LST {summary.loc["High", "mean_lst_c"]} °C vs {summary.loc["Low", "mean_lst_c"]} °C in Low')
if uhi is not None:
    print(f'Surface urban heat island (dense built-up vs green): {uhi:+.1f} °C')
print('\nPhase 3 complete. Next: Notebook 04, mapping the real September 2022 flood from Sentinel-1.')